In [1]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp, Operator
from qiskit.primitives import StatevectorEstimator

In [2]:
def create_universal_ansatz(num_qubits, num_layers):
    theta = ParameterVector("θ", num_qubits * 3 * num_layers + num_qubits * 2 * (num_layers - 1))
    ansatz = QuantumCircuit(num_qubits)
    
    param_idx = 0
    for layer in range(num_layers):
        # Single-qubit rotations: Ry, Rz on each qubit
        for q in range(num_qubits):
            ansatz.ry(theta[param_idx], q)
            param_idx += 1
            ansatz.rz(theta[param_idx], q)
            param_idx += 1
        
        # Entangling layer (CX ladder)
        if layer < num_layers - 1:
            for q in range(num_qubits - 1):
                ansatz.cx(q, q + 1)
            # Optional: wrap around
            if num_qubits > 2:
                ansatz.cx(num_qubits - 1, 0)
    
    # Final single-qubit rotations
    for q in range(num_qubits):
        ansatz.ry(theta[param_idx], q)
        param_idx += 1
        ansatz.rz(theta[param_idx], q)
        param_idx += 1
    
    return ansatz, theta

ansatz, theta = create_universal_ansatz(2, num_layers=2)
print(ansatz.draw())

     ┌──────────┐┌──────────┐     ┌──────────┐┌──────────┐ ┌──────────┐»
q_0: ┤ Ry(θ[0]) ├┤ Rz(θ[1]) ├──■──┤ Ry(θ[4]) ├┤ Rz(θ[5]) ├─┤ Ry(θ[8]) ├»
     ├──────────┤├──────────┤┌─┴─┐├──────────┤├──────────┤┌┴──────────┤»
q_1: ┤ Ry(θ[2]) ├┤ Rz(θ[3]) ├┤ X ├┤ Ry(θ[6]) ├┤ Rz(θ[7]) ├┤ Ry(θ[10]) ├»
     └──────────┘└──────────┘└───┘└──────────┘└──────────┘└───────────┘»
«      ┌──────────┐
«q_0: ─┤ Rz(θ[9]) ├
«     ┌┴──────────┤
«q_1: ┤ Rz(θ[11]) ├
«     └───────────┘


In [3]:
def hardware_efficient_ansatz(num_qubits, num_layers):
    theta = ParameterVector("θ", num_qubits * 2 * num_layers)
    ansatz = QuantumCircuit(num_qubits)
    
    param_idx = 0
    for layer in range(num_layers):
        for q in range(num_qubits):
            ansatz.ry(theta[param_idx], q)
            param_idx += 1
            ansatz.rz(theta[param_idx], q)
            param_idx += 1
        for q in range(num_qubits - 1):
            ansatz.cx(q, q + 1)
    
    return ansatz, theta

In [4]:


# Example: Hamiltonian with Ry term (or any other structure)
H = SparsePauliOp("XY") + 0.5 * SparsePauliOp("RY")  # Non-Pauli example

# Print exact spectrum
eigvals = np.linalg.eigvalsh(Operator(H).data)
print(f"Exact eigenvalues: {np.round(eigvals, 6)}")
print(f"Exact ground state energy: {eigvals[0]:.6f}\n")

# Universal 2-layer ansatz for 2 qubits
num_qubits = 2
num_layers = 2
theta = ParameterVector("θ", num_qubits * 3 * num_layers)

ansatz = QuantumCircuit(num_qubits)

param_idx = 0
for layer in range(num_layers):
    # Single-qubit rotations
    for q in range(num_qubits):
        ansatz.ry(theta[param_idx], q)
        param_idx += 1
        ansatz.rz(theta[param_idx], q)
        param_idx += 1
    
    # Entangling layer (CX)
    if layer < num_layers - 1:
        ansatz.cx(0, 1)

print("Ansatz:")
print(ansatz.draw(), "\n")

# Cost function
estimator = StatevectorEstimator()

def cost_func(params):
    job = estimator.run([(ansatz, H, [params])])
    return float(job.result()[0].data.evs[0])

# Optimize
np.random.seed(42)
best_result = None

for trial in range(8):
    x0 = np.random.uniform(0, 2 * np.pi, size=len(theta))
    res = minimize(cost_func, x0, method="cobyla", options={"maxiter": 2000})
    if best_result is None or res.fun < best_result.fun:
        best_result = res

print(f"Optimized ground state energy : {best_result.fun:.6f}")
print(f"Optimal parameters θ          : {np.round(best_result.x, 4)}")
print(f"Optimization success          : {best_result.success}")

QiskitError: 'Pauli string label "RY" is not valid.'